In [0]:
%run ./01_pipeline_config.py

In [0]:
%run ../07_rag/04_rag_retrieval.py


In [0]:
%run ../07_rag/05_rag_answer_generation.py

In [0]:
# ================================================================
# CELL 4 — IMPORTS
# ================================================================

import json
import time

print("Imports successful.")

In [0]:
# ================================================================
# PHASE 20 — PRODUCTION RAG PIPELINE
# FILE: 03_rag_pipeline.py
# ================================================================

print("=" * 70)
print("PHASE 20 — PRODUCTION RAG PIPELINE")
print("=" * 70)

In [0]:
# ================================================================
# CELL 5 — RAG DEPENDENCY VALIDATION
# ================================================================

print("=" * 70)
print("RAG DEPENDENCY VALIDATION")
print("=" * 70)

required_functions = [
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:
        failed_dependencies.append(
            function_name
        )

print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "RAG pipeline cannot continue. "
        "Missing functions: "
        + ", ".join(failed_dependencies)
    )

print()
print("RAG dependency validation: PASS")

In [0]:
# ================================================================
# CELL 6 — VERIFY RAG ANSWER GENERATOR
# ================================================================

import inspect

print("=" * 70)
print("RAG ANSWER GENERATOR VALIDATION")
print("=" * 70)

rag_source = inspect.getsource(
    generate_rag_answer
)

print(
    "Source file:",
    inspect.getsourcefile(
        generate_rag_answer
    )
)

if "dummy answer" in rag_source.lower():

    raise RuntimeError(
        "Dummy generate_rag_answer() detected."
    )

print()
print("Dummy implementation detected: NO")
print("RAG answer generator: PASS")

In [0]:
# ================================================================
# CELL 7 — PRODUCTION RAG PIPELINE FUNCTION
# ================================================================

def run_rag_pipeline(question):
    """
    Execute the complete production RAG pipeline.

    Flow:

        Question
            ↓
        Embedding
            ↓
        Retrieval
            ↓
        Context
            ↓
        LLM Answer
    """

    start_time = time.time()

    response = {
        "success": False,
        "question": question,
        "route": "rag",
        "answer": None,
        "sources": [],
        "retrieved_chunks": 0,
        "error": None,
        "execution_time_ms": None
    }

    try:

        # ====================================================
        # 1. VALIDATE QUESTION
        # ====================================================

        if not question or not str(question).strip():

            response["error"] = (
                "Question is empty."
            )

            return response

        question = str(question).strip()

        # ====================================================
        # 2. GENERATE QUESTION EMBEDDING
        # ====================================================

        question_embedding = (
            generate_question_embedding(
                question
            )
        )

        if question_embedding is None:

            response["error"] = (
                "Question embedding generation "
                "returned None."
            )

            return response

        # ====================================================
        # 3. RETRIEVE DOCUMENTS
        # ====================================================

        retrieval_results = retrieve_documents(
            question_embedding
        )

        if retrieval_results is None:

            response["error"] = (
                "Document retrieval returned None."
            )

            return response

        if not isinstance(
            retrieval_results,
            list
        ):

            response["error"] = (
                "Document retrieval returned "
                f"{type(retrieval_results).__name__} "
                "instead of a list."
            )

            return response

        response["retrieved_chunks"] = len(
            retrieval_results
        )

        # ====================================================
        # 4. BUILD RAG CONTEXT
        # ====================================================

        rag_context = build_rag_context(
            retrieval_results
        )

        if rag_context is None:

            response["error"] = (
                "RAG context generation "
                "returned None."
            )

            return response

        if not str(rag_context).strip():

            response["error"] = (
                "RAG context is empty."
            )

            return response

        # ====================================================
        # 5. GENERATE GROUNDED ANSWER
        # ====================================================

        rag_result = generate_rag_answer(
            question,
            rag_context
        )

        if rag_result is None:

            response["error"] = (
                "RAG answer generator returned None."
            )

            return response

        if not isinstance(
            rag_result,
            dict
        ):

            response["error"] = (
                "RAG answer generator returned "
                f"{type(rag_result).__name__} "
                "instead of a dictionary."
            )

            return response

        # ====================================================
        # 6. CHECK LLM RESULT
        # ====================================================

        rag_success = rag_result.get(
            "success",
            False
        )

        if not rag_success:

            response["error"] = rag_result.get(
                "error",
                "RAG answer generation failed."
            )

            return response

        response["answer"] = rag_result

        # ====================================================
        # 7. COLLECT SOURCES
        # ====================================================

        response["sources"] = [

            {
                "chunk_id": item.get(
                    "chunk_id"
                ),

                "document_id": item.get(
                    "document_id"
                ),

                "file_name": item.get(
                    "file_name"
                ),

                "title": item.get(
                    "title"
                )
            }

            for item in retrieval_results
            if isinstance(
                item,
                dict
            )
        ]

        # ====================================================
        # 8. SUCCESS
        # ====================================================

        response["success"] = True

        return response

    except Exception as e:

        response["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return response

    finally:

        response["execution_time_ms"] = round(
            (time.time() - start_time) * 1000,
            2
        )

In [0]:
# ================================================================
# CELL 8 — TEST RAG PIPELINE
# ================================================================

print("=" * 70)
print("TEST — PRODUCTION RAG PIPELINE")
print("=" * 70)

rag_test_question = (
    "What is the discount policy?"
)

rag_pipeline_result = run_rag_pipeline(
    rag_test_question
)

print(
    json.dumps(
        rag_pipeline_result,
        indent=2,
        default=str
    )
)

In [0]:
# ================================================================
# CELL 9 — DISPLAY RAG ANSWER
# ================================================================

if rag_pipeline_result["success"]:

    print("=" * 70)
    print("RAG ANSWER")
    print("=" * 70)

    rag_answer = rag_pipeline_result[
        "answer"
    ]

    print(
        rag_answer.get(
            "answer"
        )
    )

    print()
    print(
        "Retrieved chunks:",
        rag_pipeline_result[
            "retrieved_chunks"
        ]
    )

else:

    print(
        "RAG pipeline failed:"
    )

    print(
        rag_pipeline_result[
            "error"
        ]
    )

In [0]:
# ================================================================
# CELL 10 — SOURCE VALIDATION
# ================================================================

print("=" * 70)
print("RAG SOURCE VALIDATION")
print("=" * 70)

sources = rag_pipeline_result.get(
    "sources",
    []
)

print(
    "Sources returned:",
    len(sources)
)

for source in sources:

    print(
        f"- {source.get('file_name')} "
        f"— {source.get('title')}"
    )

if rag_pipeline_result["success"]:

    if len(sources) == 0:

        raise RuntimeError(
            "RAG succeeded but returned "
            "no document sources."
        )

print()
print("Source validation: PASS")

In [0]:
# ================================================================
# CELL 11 — DUMMY ANSWER PROTECTION
# ================================================================

print("=" * 70)
print("RAG ANSWER QUALITY CHECK")
print("=" * 70)

answer_text = str(
    rag_pipeline_result.get(
        "answer"
    )
)

if "dummy answer" in answer_text.lower():

    raise RuntimeError(
        "Dummy answer detected in production RAG pipeline."
    )

print(
    "Dummy answer detected: NO"
)

print(
    "Grounded answer detected: YES"
)

In [0]:
# ================================================================
# CELL 12 — FINAL RAG PIPELINE VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 20 — RAG PIPELINE VALIDATION")
print("=" * 70)

rag_checks = [

    (
        "Pipeline succeeded",
        rag_pipeline_result.get(
            "success"
        ) is True
    ),

    (
        "Answer returned",
        rag_pipeline_result.get(
            "answer"
        ) is not None
    ),

    (
        "Retrieved documents",
        rag_pipeline_result.get(
            "retrieved_chunks",
            0
        ) > 0
    ),

    (
        "Sources returned",
        len(
            rag_pipeline_result.get(
                "sources",
                []
            )
        ) > 0
    ),

    (
        "No error",
        rag_pipeline_result.get(
            "error"
        ) is None
    )
]

failed_checks = []

for check_name, passed in rag_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{check_name}"
    )

    if not passed:

        failed_checks.append(
            check_name
        )

print()
print(
    "Total checks:",
    len(rag_checks)
)

print(
    "Failed checks:",
    len(failed_checks)
)

if failed_checks:

    raise RuntimeError(
        "RAG pipeline validation failed: "
        + ", ".join(failed_checks)
    )

print()
print(
    "PHASE 20 RAG PIPELINE: PASS ✓"
)

In [0]:
# ================================================================
# CELL 13 — RAG PIPELINE SUMMARY
# ================================================================

print("=" * 70)
print("PRODUCTION RAG PIPELINE SUMMARY")
print("=" * 70)

print()
print(
    "Question:",
    rag_pipeline_result["question"]
)

print(
    "Route:",
    rag_pipeline_result["route"]
)

print(
    "Retrieved chunks:",
    rag_pipeline_result["retrieved_chunks"]
)

print(
    "Sources:",
    len(
        rag_pipeline_result["sources"]
    )
)

print(
    "Execution time:",
    rag_pipeline_result[
        "execution_time_ms"
    ],
    "ms"
)

print(
    "Success:",
    rag_pipeline_result["success"]
)

print()
print("=" * 70)
print("RAG PIPELINE STATUS: PASS ✓")
print("=" * 70)